In [1]:
from transformers import AutoTokenizer
# 1、传入模型目录，加载tokenizer
model_path = "model/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2、构造一个demo 的 message_list
message_list = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好，我是智能助手"}
]

# 3、对数据，使用chat template 进行格式化
res = tokenizer.apply_chat_template(message_list,tokenize=False)

In [4]:
print(res)
# 得到的res就是用以训练的 多轮对话的字符串表达形式

<|im_start|>user
你好<|im_end|>
<|im_start|>assistant
<think>

</think>

你好，我是智能助手<|im_end|>



In [6]:
# 3、推理时，同样需要使用和训练时相同的chat template，对message_list进行格式化
message_list2 = [
    {"role":"system","content":"你是一个专业的翻译"},
    {"role":"user","content":"你好，帮我把英文翻译成中文：hello"},
]

model_input_str = tokenizer.apply_chat_template(message_list2,tokenize=False,add_generation_prompt=True)

In [7]:
print(model_input_str)

<|im_start|>system
你是一个专业的翻译<|im_end|>
<|im_start|>user
你好，帮我把英文翻译成中文：hello<|im_end|>
<|im_start|>assistant



In [8]:
# 推理的时候，开启thinking模式，和不开启thinking模式
# 上述这种模式，模型接下来会预测<think>内容，所以就是开启thinking模式
# 不开启thinking模式：
model_input_str_no_thinking = tokenizer.apply_chat_template(message_list2,tokenize=False,add_generation_prompt=True,enable_thinking=False)

In [9]:
print(model_input_str_no_thinking)

<|im_start|>system
你是一个专业的翻译<|im_end|>
<|im_start|>user
你好，帮我把英文翻译成中文：hello<|im_end|>
<|im_start|>assistant
<think>

</think>




In [10]:
# 使用chat template，对数据集当中的数据进行处理
def get_train_data():

    from datasets import load_dataset
    dataset = load_dataset("data/ultrachat_200k")
    train_sft_data:list = dataset["train_sft"]
    train_data = []
    for i in range(100):
        message_list = train_sft_data[i]["messages"]
        tokenized_data = tokenizer.apply_chat_template(message_list,tokenize=True,truncation=True,max_length = 2500)
        train_data.append(tokenized_data)
    
    return train_data

In [12]:
res = get_train_data()

In [13]:
len(res)

100

In [17]:
for key, value in res[0].items():
    print(key)
    print(value)

input_ids
[151644, 872, 198, 9485, 11221, 3796, 311, 3772, 5980, 21386, 320, 78795, 220, 21, 13, 15, 44662, 10392, 2210, 220, 19, 13, 15, 44662, 4270, 44168, 220, 18, 13, 15, 10, 47175, 220, 17, 13, 15, 44662, 34906, 24078, 220, 20, 13, 15, 10, 568, 3555, 6912, 2319, 1079, 358, 1667, 5267, 1925, 697, 25326, 6816, 609, 50219, 25326, 14158, 11, 498, 646, 6707, 1473, 279, 14246, 2168, 315, 264, 1985, 389, 19548, 553, 27362, 825, 315, 279, 6912, 594, 5798, 3419, 5003, 4894, 7771, 11101, 6816, 609, 50219, 25326, 14158, 686, 1431, 3037, 279, 14246, 1985, 2168, 1101, 553, 68607, 916, 429, 1985, 2168, 28574, 624, 21468, 419, 4565, 3796, 311, 678, 14158, 315, 279, 6912, 476, 1101, 3151, 6174, 438, 10007, 304, 279, 1467, 3684, 30, 151645, 198, 151644, 77091, 198, 1986, 4565, 1172, 16790, 311, 11101, 6816, 323, 50219, 25326, 14158, 315, 279, 3772, 5980, 21386, 10007, 304, 279, 1467, 3684, 13, 151645, 198, 151644, 872, 198, 6713, 498, 8474, 752, 1526, 279, 1882, 315, 27362, 279, 14246, 2168, 19548